In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

arquivo = "/content/drive/MyDrive/0 - Senai Assistente IA 2026.1/Aula 08/datasetExamesSangue10K.csv"

datasetExames = pd.read_csv(arquivo)
print("Dataset carregado com sucesso!")

print(datasetExames.head())

print("\nQuantidade por diagnóstico:")
print(datasetExames["diagnostico"].value_counts())

dadosEntrada = datasetExames[[
        "idade","hemoglobina","hematocrito","hemacias","leucocitos",
        "plaquetas","glicemia","pressaoSistolica","pressaoDiastolica"]]

dadosSaidaTexto = datasetExames["diagnostico"]
encoderDiagnostico = LabelEncoder()
dadosSaidaNumero = encoderDiagnostico.fit_transform(dadosSaidaTexto)
dadosSaida = to_categorical(dadosSaidaNumero)
print("\nClasses do modelo:")
print(encoderDiagnostico.classes_)

dadosTreinoEntrada, dadosTesteEntrada,dadosTreinoSaida, dadosTesteSaida = train_test_split(dadosEntrada,
    dadosSaida, test_size=0.20, random_state=42)

normalizador = StandardScaler()
dadosTreinoEntradaNormalizados = normalizador.fit_transform(dadosTreinoEntrada)
dadosTesteEntradaNormalizados = normalizador.transform(dadosTesteEntrada)

modeloRedeNeural = Sequential()
modeloRedeNeural.add(Dense(16,activation="relu",input_shape=(9,)))
#modeloRedeNeural.add(Dense(12,activation="relu"))
#modeloRedeNeural.add(Dense(10,activation="relu"))
#modeloRedeNeural.add(Dense(8,activation="relu"))
#modeloRedeNeural.add(Dense(6,activation="relu"))
modeloRedeNeural.add(Dense(4,activation="softmax"))

modeloRedeNeural.compile(optimizer="adam",
                         loss="categorical_crossentropy",
                         metrics=["accuracy"])

historico = modeloRedeNeural.fit(dadosTreinoEntradaNormalizados,
                                 dadosTreinoSaida,
                                 epochs=50,
                                 batch_size=32,
                                 validation_split=0.20,
                                 verbose=1)

perda, acuracia = modeloRedeNeural.evaluate(
    dadosTesteEntradaNormalizados,
    dadosTesteSaida,
    verbose=0)

print(f"Erro do modelo (Loss): {perda:.4f}")
print(f"Taxa de acerto (Accuracy): {acuracia:.2%}")

print(f"Erro do modelo (Loss): {perda:.4f}")
print(f"Taxa de acerto (Accuracy): {acuracia:.2%}")

novoPaciente = pd.DataFrame({
    "idade": [58],
    "hemoglobina": [14.2],
    "hematocrito": [42],
    "hemacias": [4.8],
    "leucocitos": [7000],
    "plaquetas": [260],
    "glicemia": [210],
    "pressaoSistolica": [120],
    "pressaoDiastolica": [80]
    })

novoPacienteNormalizado = normalizador.transform(novoPaciente)
probabilidades = modeloRedeNeural.predict(novoPacienteNormalizado)
indiceClasse = probabilidades.argmax()
diagnostico = encoderDiagnostico.inverse_transform([indiceClasse])
print("Diagnóstico previsto:", diagnostico[0])

print("\nProbabilidades:")

for i in range(len(encoderDiagnostico.classes_)):
    print(encoderDiagnostico.classes_[i],":",round(probabilidades[0][i] * 100, 2),"%")